In [1]:
import os
from getpass import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith API Key: ")

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

print(f"Loaded {len(blog_docs)} documents")

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_24157/3783885917.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 documents


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

splits = text_splitter.split_documents(blog_docs)

print(f"Created {len(splits)} chunks")

Created 50 chunks


In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

print("Vector store created successfully!")

Vector store created successfully!


In [5]:
question = "What is agent memory?"

retrieved_docs = retriever.invoke(question)

print(f"Retrieved {len(retrieved_docs)} documents")

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content[:300])

Retrieved 4 documents

Document 1:
Memory

Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.
Long-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging 

Document 2:
Long-Term Memory (LTM): Long-term memory can store information for a remarkably long time, ranging from a few days to decades, with an essentially unlimited storage capacity. There are two subtypes of LTM:

Explicit / declarative memory: This is memory of facts and events, and refers to those memori

Document 3:
Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with this limited communication bandwidth, while mechanisms like self-reflection to learn from past mist

Document 4:
Each element is an obse

In [6]:
from langchain_core.prompts import ChatPromptTemplate

template = """You are a helpful assistant that generates multiple
search queries based on a single input query.

Generate multiple search queries related to: {question}

Output exactly 4 queries, one per line.
Do not number the queries or add explanations."""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [7]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

generate_queries = (
    prompt_rag_fusion
    | llm
    | StrOutputParser()
    | (lambda x: [
        line.strip()
        for line in x.split("\n")
        if line.strip()
    ])
)

In [8]:
question = "What is task decomposition for LLM agents?"

queries = generate_queries.invoke({
    "question": question
})

for i, query in enumerate(queries, 1):
    print(f"Query {i}: {query}")

Query 1: What is task decomposition for LLM agents?
Query 2: How does task decomposition work for large language model agents?
Query 3: Task decomposition for LLM agents: a review of the current state
Query 4: Task decomposition in LLM agents: a method for improving performance


In [9]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):

    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):

            doc_str = dumps(doc)

            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0

            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(
            fused_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )
    ]

    return reranked_results

In [10]:
question = "What is task decomposition for LLM agents?"

retrieval_chain_rag_fusion = (
    generate_queries
    | retriever.map()
    | reciprocal_rank_fusion
)

docs = retrieval_chain_rag_fusion.invoke({
    "question": question
})

print(f"Total unique documents: {len(docs)}")

Total unique documents: 5


/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_24157/3661687149.py:18: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  (loads(doc), score)
/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_24157/3661687149.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


In [11]:
for i, (doc, score) in enumerate(docs[:5], 1):
    print(f"\nRank {i}")
    print(f"RRF Score: {score:.5f}")
    print(f"Content: {doc.page_content[:250]}")
    print("-" * 50)


Rank 1
RRF Score: 0.06613
Content: LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as Aut
--------------------------------------------------

Rank 2
RRF Score: 0.06478
Content: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model pe
--------------------------------------------------

Rank 3
RRF Score: 0.06429
Content: Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with this limited communication bandwidth, while mecha
-------------------------------------

In [12]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [13]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

def format_docs(ranked_docs):
    return "\n\n".join(
        doc.page_content
        for doc, score in ranked_docs[:5]
    )

final_rag_chain = (
    {
        "context": retrieval_chain_rag_fusion | format_docs,
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [15]:
question = "What is task decomposition for LLM agents?"

response = final_rag_chain.invoke({
    "question": question
})

print(response)

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_24157/3661687149.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


Task decomposition for LLM (Large Language Model) agents is the process of breaking down complex tasks into smaller, manageable subgoals. This is done to enable efficient handling of complex tasks and to improve the quality of final results. In the context of LLM-powered autonomous agent systems, task decomposition involves using techniques such as Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023) to decompose big tasks into multiple simpler steps.


In [16]:
question = "What is task decomposition for LLM agents?"

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 10}
)

retrieved_docs = retriever.invoke(question)

print(f"Retrieved {len(retrieved_docs)} documents")

Retrieved 10 documents


In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# Create question-document pairs
pairs = [
    (question, doc.page_content)
    for doc in retrieved_docs
]

# Calculate relevance scores
scores = reranker.predict(pairs)

# Combine documents and scores
ranked_docs = sorted(
    zip(retrieved_docs, scores),
    key=lambda x: float(x[1]),
    reverse=True
)

# Display top 3 results
for i, (doc, score) in enumerate(ranked_docs[:3], 1):
    print(f"\nRank {i}")
    print(f"Score: {float(score):.4f}")
    print(f"Content: {doc.page_content[:300]}")
    print("-" * 50)

/Users/aish/Desktop/rag_from_scratch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1111 23:58:45.761000 24157 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11159.59it/s]



Rank 1
Score: 3.7921
Content: LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring
--------------------------------------------------

Rank 2
Score: 2.9235
Content: Finite context length: The restricted context capacity limits the inclusion of historical information, detailed instructions, API call context, and responses. The design of the system has to work with this limited communication bandwidth, while mechanisms like self-reflection to learn from past mist
--------------------------------------------------

Rank 3
Score: 2.5504
Content: Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a stan

In [18]:
context = "\n\n".join(
    doc.page_content
    for doc, score in ranked_docs[:3]
)

response = (
    prompt
    | llm
    | StrOutputParser()
).invoke({
    "context": context,
    "question": question
})

print(response)

Task decomposition for LLM agents is the process of breaking down a large task into smaller, manageable subgoals. This allows the agent to efficiently handle complex tasks by identifying the individual steps required to achieve the overall goal.


## Retrieval (CRAG)

In [19]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# Define the expected output
class GradeDocuments(BaseModel):
    binary_score: str = Field(
        description="Document relevance: yes or no"
    )

# Initialize Ollama
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

structured_llm = llm.with_structured_output(
    GradeDocuments
)

# Create grading prompt
system = """You are a grader assessing relevance
of a retrieved document to a user question.

If the document contains information related to
the question, grade it as relevant.

Give a binary score of 'yes' or 'no'."""

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "Retrieved document:\n{document}\n\nUser question: {question}")
])

retrieval_grader = grade_prompt | structured_llm

In [20]:
question = "What is task decomposition for LLM agents?"

docs = retriever.invoke(question)

doc_txt = docs[0].page_content

score = retrieval_grader.invoke({
    "question": question,
    "document": doc_txt
})

print("Relevance score:", score.binary_score)

Relevance score: yes


In [21]:
def grade_documents(question, documents):
    filtered_docs = []
    irrelevant_docs = []
    web_search = "No"

    for i, doc in enumerate(documents):

        score = retrieval_grader.invoke({
            "question": question,
            "document": doc.page_content
        })

        grade = score.binary_score.strip().lower()

        if grade == "yes":
            print(f"Document {i+1}: RELEVANT")
            filtered_docs.append(doc)

        else:
            print(f"Document {i+1}: NOT RELEVANT")
            irrelevant_docs.append(doc)
            web_search = "Yes"

    return filtered_docs, irrelevant_docs, web_search

In [22]:
question = "What is task decomposition for LLM agents?"

docs = retriever.invoke(question)

filtered_docs, irrelevant_docs, web_search = grade_documents(
    question,
    docs
)

print("\n--- RESULTS ---")
print("Total documents:", len(docs))
print("Relevant documents:", len(filtered_docs))
print("Irrelevant documents:", len(irrelevant_docs))
print("Web search needed:", web_search)

Document 1: RELEVANT
Document 2: NOT RELEVANT
Document 3: RELEVANT
Document 4: RELEVANT
Document 5: NOT RELEVANT
Document 6: RELEVANT
Document 7: NOT RELEVANT
Document 8: NOT RELEVANT
Document 9: NOT RELEVANT
Document 10: NOT RELEVANT

--- RESULTS ---
Total documents: 10
Relevant documents: 4
Irrelevant documents: 6
Web search needed: Yes


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rewrite_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a question rewriter.

        Rewrite the user's question to make it better
        for retrieving relevant documents.

        Preserve the original meaning.

        Return only the rewritten question."""
    ),
    ("human", "Original question: {question}")
])

question_rewriter = (
    rewrite_prompt
    | llm
    | StrOutputParser()
)

In [24]:
question = "How do agents remember things?"

better_question = question_rewriter.invoke({
    "question": question
})

print("Original question:", question)
print("Rewritten question:", better_question)

Original question: How do agents remember things?
Rewritten question: How do law enforcement or private investigation agents retain and recall specific details, such as witness descriptions, suspect information, or case evidence, over time?


In [25]:
question = "What is task decomposition for LLM agents?"

docs = retriever.invoke(question)

filtered_docs, irrelevant_docs, web_search = grade_documents(
    question,
    docs
)

if web_search == "Yes":

    print("\nCorrective retrieval needed.")

    better_question = question_rewriter.invoke({
        "question": question
    })

    print("Rewritten question:", better_question)

    new_docs = retriever.invoke(better_question)

    print("New documents retrieved:", len(new_docs))

else:
    print("\nDocuments passed the relevance check.")

Document 1: RELEVANT
Document 2: NOT RELEVANT
Document 3: RELEVANT
Document 4: RELEVANT
Document 5: NOT RELEVANT
Document 6: RELEVANT
Document 7: NOT RELEVANT
Document 8: NOT RELEVANT
Document 9: NOT RELEVANT
Document 10: NOT RELEVANT

Corrective retrieval needed.
Rewritten question: Rewritten question: Can you explain the concept of task decomposition in the context of Large Language Model (LLM) agents, including its purpose, benefits, and common techniques used to implement it?
New documents retrieved: 10


In [26]:
# Evaluate newly retrieved documents
new_filtered_docs, new_irrelevant_docs, _ = grade_documents(
    question,
    new_docs
)

print("\n--- CORRECTIVE RETRIEVAL RESULTS ---")
print("Newly retrieved:", len(new_docs))
print("Relevant:", len(new_filtered_docs))
print("Irrelevant:", len(new_irrelevant_docs))

Document 1: RELEVANT
Document 2: RELEVANT
Document 3: RELEVANT
Document 4: NOT RELEVANT
Document 5: NOT RELEVANT
Document 6: RELEVANT
Document 7: NOT RELEVANT
Document 8: NOT RELEVANT
Document 9: NOT RELEVANT
Document 10: NOT RELEVANT

--- CORRECTIVE RETRIEVAL RESULTS ---
Newly retrieved: 10
Relevant: 4
Irrelevant: 6


In [27]:
# Combine relevant documents
all_relevant_docs = filtered_docs + new_filtered_docs

# Remove duplicates
unique_docs = {}

for doc in all_relevant_docs:
    unique_docs[doc.page_content] = doc

final_docs = list(unique_docs.values())

print("Total unique relevant documents:", len(final_docs))

Total unique relevant documents: 4


In [29]:
print("Question:", repr(question))
print("Number of relevant documents:", len(final_docs))
print("Context length:", len(context))

Question: 'What is task decomposition for LLM agents?'
Number of relevant documents: 4
Context length: 4956


In [30]:
messages = final_prompt.format_messages(
    context=context,
    question=question
)

for message in messages:
    print(f"\n{message.type.upper()}:")
    print(message.content[:500])


HUMAN:
Answer the question using only the provided context.
    If the context does not contain enough information,
    say that you don't have enough information.

    Context:
    LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiali


In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

final_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant answering questions
        about LLM-powered autonomous agents.

        The user's question is:
        {question}

        Answer the question using the provided context.
        Do not ask the user to provide a question.
        If the answer is not available in the context,
        say that the context does not contain enough information."""
    ),
    (
        "human",
        """Question: {question}

        Relevant context:
        {context}

        Now answer the question: {question}"""
    )
])

final_chain = final_prompt | llm | StrOutputParser()

In [33]:
question = "What is task decomposition for LLM agents?"

context = "\n\n".join(
    doc.page_content
    for doc in final_docs[:4]
)

answer = final_chain.invoke({
    "question": question,
    "context": context
})

print(answer)

Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This process enables efficient handling of complex tasks. The LLM-powered autonomous agent system uses a technique called Chain of Thought (CoT) or Tree of Thoughts to decompose big tasks into multiple simpler steps. The model is instructed to "think step by step" to utilize more test-time computation and shed light into the model's thinking process. This technique transforms complex tasks into multiple manageable tasks, providing an interpretation of the model's thinking process.


## Self-RAG

In [34]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class GradeHallucinations(BaseModel):
    binary_score: Literal["yes", "no"] = Field(
        description="Is the answer supported by the documents?"
    )

structured_llm_grader = llm.with_structured_output(
    GradeHallucinations
)

hallucination_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a grader assessing whether an LLM
        answer is grounded in a set of retrieved documents.

        Give 'yes' if the answer is supported by
        the documents.

        Give 'no' if the answer contains unsupported
        factual claims."""
    ),
    (
        "human",
        """Retrieved documents:
        {documents}

        Generated answer:
        {generation}"""
    )
])

hallucination_grader = (
    hallucination_prompt
    | structured_llm_grader
)

In [35]:
documents_text = "\n\n".join(
    doc.page_content
    for doc in final_docs
)

score = hallucination_grader.invoke({
    "documents": documents_text,
    "generation": answer
})

print("Grounded in documents:", score.binary_score)

Grounded in documents: yes


In [36]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate

class GradeAnswer(BaseModel):
    binary_score: Literal["yes", "no"] = Field(
        description="Does the answer address the question?"
    )

structured_llm_answer = llm.with_structured_output(
    GradeAnswer
)

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a grader assessing whether
        an LLM-generated answer addresses a user's question.

        Give 'yes' if the answer directly addresses
        the question.

        Give 'no' if the answer is irrelevant,
        incomplete, or does not answer the question."""
    ),
    (
        "human",
        """User question:
        {question}

        Generated answer:
        {generation}"""
    )
])

answer_grader = answer_prompt | structured_llm_answer

In [37]:
question = "What is task decomposition for LLM agents?"

score = answer_grader.invoke({
    "question": question,
    "generation": answer
})

print("Answers the question:", score.binary_score)

Answers the question: yes


In [38]:
def self_rag(question, max_retries=2):

    print("\n--- STEP 1: RETRIEVING DOCUMENTS ---")

    documents = retriever.invoke(question)

    print(f"Retrieved {len(documents)} documents")

    # Grade the retrieved documents
    filtered_docs, _, _ = grade_documents(
        question,
        documents
    )

    if not filtered_docs:
        return "No relevant documents found."

    # Prepare context
    context = "\n\n".join(
        doc.page_content
        for doc in filtered_docs[:5]
    )

    for attempt in range(max_retries + 1):

        print(f"\n--- GENERATION ATTEMPT {attempt + 1} ---")

        # Generate answer
        generation = final_chain.invoke({
            "context": context,
            "question": question
        })

        # Check whether answer is grounded
        hallucination_score = hallucination_grader.invoke({
            "documents": context,
            "generation": generation
        })

        grounded = (
            hallucination_score.binary_score.strip().lower()
            == "yes"
        )

        print("Grounded in documents:", grounded)

        # Check whether answer addresses the question
        answer_score = answer_grader.invoke({
            "question": question,
            "generation": generation
        })

        useful = (
            answer_score.binary_score.strip().lower()
            == "yes"
        )

        print("Answers the question:", useful)

        if grounded and useful:
            print("\nAnswer accepted!")
            return generation

        print("Answer did not pass both checks. Retrying...")

    return (
        "The system could not generate an answer "
        "that passed both evaluation checks."
    )

In [39]:
question = "What is task decomposition for LLM agents?"

response = self_rag(question)

print("\n--- FINAL ANSWER ---")
print(response)


--- STEP 1: RETRIEVING DOCUMENTS ---
Retrieved 10 documents
Document 1: RELEVANT
Document 2: NOT RELEVANT
Document 3: RELEVANT
Document 4: RELEVANT
Document 5: NOT RELEVANT
Document 6: RELEVANT
Document 7: NOT RELEVANT
Document 8: NOT RELEVANT
Document 9: NOT RELEVANT
Document 10: NOT RELEVANT

--- GENERATION ATTEMPT 1 ---
Grounded in documents: True
Answers the question: True

Answer accepted!

--- FINAL ANSWER ---
Task decomposition for LLM agents involves breaking down large tasks into smaller, manageable subgoals. This process enables efficient handling of complex tasks. The LLM-powered autonomous agent system uses a technique called Chain of Thought (CoT) or Tree of Thoughts to decompose big tasks into multiple simpler steps. The model is instructed to "think step by step" to utilize more test-time computation and shed light into the model's thinking process. This technique transforms complex tasks into multiple manageable tasks, providing an interpretation of the model's thinkin

## Impact of Long Context

In [40]:
question = "What is task decomposition for LLM agents?"

# Retrieve 10 documents
docs = retriever.invoke(question)

# Short context: first 2 documents
short_context = "\n\n".join(
    doc.page_content for doc in docs[:2]
)

# Long context: all 10 documents
long_context = "\n\n".join(
    doc.page_content for doc in docs
)

print("Short context length:", len(short_context))
print("Long context length:", len(long_context))

Short context length: 1103
Long context length: 10775


In [41]:
short_answer = final_chain.invoke({
    "context": short_context,
    "question": question
})

long_answer = final_chain.invoke({
    "context": long_context,
    "question": question
})

print("=== SHORT CONTEXT ANSWER ===")
print(short_answer)

print("\n=== LONG CONTEXT ANSWER ===")
print(long_answer)

=== SHORT CONTEXT ANSWER ===
Task decomposition for LLM agents refers to the process of breaking down large tasks into smaller, manageable subgoals. This enables the efficient handling of complex tasks and allows the agent to focus on one subgoal at a time, making it more manageable and less overwhelming.

=== LONG CONTEXT ANSWER ===
Task decomposition for LLM agents is the process of breaking down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks. This is achieved through techniques such as Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023), which transform big tasks into multiple simpler steps. The goal is to utilize more test-time computation to decompose hard tasks into smaller and simpler steps, allowing the LLM to efficiently handle complex tasks.


In [42]:
question = "What is the secret project code?"

relevant_text = (
    "The secret project code is ORION-742. "
    "This is the official internal project identifier."
)

filler = (
    "This paragraph discusses general topics related to "
    "AI systems, machine learning, software development, "
    "and data processing. It does not contain any "
    "project identification codes. "
) * 35

beginning_context = relevant_text + "\n\n" + filler

middle_context = (
    filler[:len(filler)//2]
    + "\n\n" + relevant_text + "\n\n"
    + filler[len(filler)//2:]
)

end_context = filler + "\n\n" + relevant_text

print("Beginning length:", len(beginning_context))
print("Middle length:", len(middle_context))
print("End length:", len(end_context))

Beginning length: 6319
Middle length: 6321
End length: 6319


In [43]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

test_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the question using only the provided context.
        Return only the project code.
        If the code is not found, return NOT FOUND."""
    ),
    (
        "human",
        """Context:
        {context}

        Question: {question}"""
    )
])

test_chain = test_prompt | llm | StrOutputParser()

contexts = {
    "Beginning": beginning_context,
    "Middle": middle_context,
    "End": end_context
}

for position, ctx in contexts.items():
    response = test_chain.invoke({
        "context": ctx,
        "question": question
    })

    print(f"{position}: {response}")

Beginning: ORION-742
Middle: ORION-742
End: ORION-742
